In [1]:
# ============================================================
# Update all v3 voxel phase-separation metadata using the
# Seitz nbins rule.
#
# This version bypasses the buggy
# write_voxel_phase_separation_metadata wrapper.
# ============================================================

from pathlib import Path
import importlib

import gsd.hoomd
import numpy as np
import pandas as pd

from md_Helpers import classification
from md_Helpers import metadata

from md_Helpers.paths import (
    THERMALIZED_STATES_V3_ROOT,
    CAVITATION_EVOLVED_V3_ROOT,
    EXCITATION_EVOLVED_V3_ROOT,
    MASTER_CSVS_V3_ROOT,
)

importlib.reload(classification)
importlib.reload(metadata)

print("Classification module:")
print(classification.__file__)


# ============================================================
# Settings
# ============================================================

DRY_RUN = False

DENSITY_THRESHOLD = (
    classification.DEFAULT_PHASE_SEP_DENSITY_THRESHOLD
)

VOXEL_FRACTION_THRESHOLD = (
    classification.DEFAULT_PHASE_SEP_VOXEL_FRACTION_THRESHOLD
)

simulation_roots = {
    "thermalized": Path(
        THERMALIZED_STATES_V3_ROOT
    ),
    "cavitation_evolved": Path(
        CAVITATION_EVOLVED_V3_ROOT
    ),
    "excitation_evolved": Path(
        EXCITATION_EVOLVED_V3_ROOT
    ),
}

report_path = (
    Path(MASTER_CSVS_V3_ROOT)
    / "voxel_phase_reclassification_seitz_nbins.csv"
)

report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Helpers
# ============================================================

def seitz_nbins(n_fcc_cells):
    return round(
        0.3 * int(n_fcc_cells) + 3
    )


def find_saved_state(log_path, paths_metadata):
    """
    Find a usable final state or trajectory for a log.
    """

    candidates = []

    # Prefer paths explicitly recorded in metadata.
    for key in [
        "final_state_path",
        "state_path",
        "trajectory_path",
    ]:
        value = paths_metadata.get(key)

        if value:
            candidates.append(Path(value))

    # Add filename-based fallbacks.
    candidates.extend(
        [
            log_path.with_name(
                log_path.name.replace(
                    "_log.hdf5",
                    ".gsd",
                )
            ),
            log_path.with_name(
                log_path.name.replace(
                    "_log.hdf5",
                    "_final.gsd",
                )
            ),
            log_path.with_name(
                log_path.name.replace(
                    "_log.hdf5",
                    "_trajectory.gsd",
                )
            ),
        ]
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


# ============================================================
# Find all logs
# ============================================================

log_records = []

for simulation_kind, root in simulation_roots.items():
    if not root.exists():
        print("Missing root:", root)
        continue

    paths = sorted(
        root.glob("**/*_log.hdf5")
    )

    print(
        f"{simulation_kind}: {len(paths)} logs"
    )

    for log_path in paths:
        log_records.append(
            {
                "simulation_kind": simulation_kind,
                "log_path": log_path,
            }
        )

print("\nTotal logs:", len(log_records))

if not log_records:
    raise RuntimeError(
        "No v3 log files were found."
    )


# ============================================================
# Process every log
# ============================================================

report_rows = []

for index, record in enumerate(
    log_records,
    start=1,
):
    simulation_kind = record["simulation_kind"]
    log_path = Path(record["log_path"])

    row = {
        "status": "failed",
        "simulation_kind": simulation_kind,
        "n_fcc_cells": np.nan,
        "target_rho": np.nan,
        "actual_rho": np.nan,
        "kT": np.nan,
        "expected_nbins": np.nan,
        "old_nbins": np.nan,
        "new_nbins": np.nan,
        "old_nbins_source": "",
        "new_nbins_source": "",
        "old_phase_separated": np.nan,
        "new_phase_separated": np.nan,
        "old_low_density_fraction": np.nan,
        "new_low_density_fraction": np.nan,
        "phase_classification_changed": False,
        "state_path": "",
        "log_path": str(log_path),
        "error": "",
    }

    try:
        # --------------------------------------------------------
        # Read state and path metadata
        # --------------------------------------------------------

        state_metadata = metadata.read_attrs(
            log_path,
            "metadata/state",
        )

        paths_metadata = metadata.read_attrs(
            log_path,
            "metadata/paths",
        )

        n_fcc_cells = state_metadata.get(
            "n_fcc_cells",
            None,
        )

        if n_fcc_cells is None:
            raise KeyError(
                "metadata/state does not contain n_fcc_cells"
            )

        n_fcc_cells = int(n_fcc_cells)
        nbins = seitz_nbins(n_fcc_cells)

        row.update(
            {
                "n_fcc_cells": n_fcc_cells,
                "target_rho": state_metadata.get(
                    "target_rho",
                    np.nan,
                ),
                "actual_rho": state_metadata.get(
                    "actual_rho",
                    np.nan,
                ),
                "kT": state_metadata.get(
                    "kT",
                    np.nan,
                ),
                "expected_nbins": nbins,
            }
        )

        # --------------------------------------------------------
        # Locate the saved GSD state
        # --------------------------------------------------------

        state_path = find_saved_state(
            log_path,
            paths_metadata,
        )

        if state_path is None:
            raise FileNotFoundError(
                f"Could not find a saved state or trajectory "
                f"for log: {log_path}"
            )

        row["state_path"] = str(state_path)

        # --------------------------------------------------------
        # Read old classification
        # --------------------------------------------------------

        old_voxel, _ = (
            classification.read_phase_method_attrs(
                log_path,
                "voxel",
            )
        )

        old_phase = old_voxel.get(
            "phase_separated",
            None,
        )

        row.update(
            {
                "old_nbins": old_voxel.get(
                    "nbins",
                    np.nan,
                ),
                "old_nbins_source": old_voxel.get(
                    "nbins_source",
                    "",
                ),
                "old_phase_separated": (
                    old_phase
                    if old_phase is not None
                    else np.nan
                ),
                "old_low_density_fraction": (
                    old_voxel.get(
                        "low_density_fraction",
                        np.nan,
                    )
                ),
            }
        )

        # --------------------------------------------------------
        # Load final frame and classify with Seitz nbins
        # --------------------------------------------------------

        with gsd.hoomd.open(
            name=str(state_path),
            mode="r",
        ) as trajectory:
            if len(trajectory) == 0:
                raise ValueError(
                    f"GSD contains no frames: {state_path}"
                )

            final_frame = trajectory[-1]

            new_voxel = (
                classification
                .compute_voxel_fraction_phase_separation(
                    final_frame,
                    nbins=nbins,
                    density_threshold=DENSITY_THRESHOLD,
                    voxel_fraction_threshold=(
                        VOXEL_FRACTION_THRESHOLD
                    ),
                )
            )

        # Explicitly record why this nbins was selected.
        new_voxel = dict(new_voxel)

        new_voxel["nbins"] = int(nbins)
        new_voxel["nbins_source"] = (
            "n_fcc_cells_rule"
        )
        new_voxel["n_fcc_cells"] = int(
            n_fcc_cells
        )
        new_voxel["updated_from_saved_gsd"] = True

        new_phase = bool(
            new_voxel["phase_separated"]
        )

        # --------------------------------------------------------
        # Write canonical v3 metadata
        # --------------------------------------------------------

        if not DRY_RUN:
            classification.write_phase_method_metadata(
                log_path=log_path,
                method_name="voxel",
                attrs=new_voxel,
                set_main_phase_separated=new_phase,
            )

        row.update(
            {
                "status": (
                    "dry_run"
                    if DRY_RUN
                    else "updated"
                ),
                "new_nbins": new_voxel["nbins"],
                "new_nbins_source": new_voxel[
                    "nbins_source"
                ],
                "new_phase_separated": new_phase,
                "new_low_density_fraction": (
                    new_voxel[
                        "low_density_fraction"
                    ]
                ),
                "phase_classification_changed": (
                    old_phase != new_phase
                ),
            }
        )

    except Exception as error:
        row["status"] = "failed"
        row["error"] = repr(error)

    report_rows.append(row)

    if (
        index % 25 == 0
        or index == len(log_records)
    ):
        failures = sum(
            item["status"] == "failed"
            for item in report_rows
        )

        print(
            f"Processed {index}/{len(log_records)} "
            f"(failures: {failures})"
        )


# ============================================================
# Audit report
# ============================================================

reclassification_report = pd.DataFrame(
    report_rows
)

reclassification_report.to_csv(
    report_path,
    index=False,
)

successful_mask = (
    reclassification_report["status"]
    != "failed"
)

failed_mask = (
    reclassification_report["status"]
    == "failed"
)

changed_mask = (
    successful_mask
    & reclassification_report[
        "phase_classification_changed"
    ].fillna(False).astype(bool)
)

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print("\nStatus counts:")
print(
    reclassification_report["status"]
    .value_counts(dropna=False)
)

print(
    "\nSuccessfully processed:",
    int(successful_mask.sum()),
)

print(
    "Classifications changed:",
    int(changed_mask.sum()),
)

print(
    "Failures:",
    int(failed_mask.sum()),
)

print("\nReport saved to:")
print(report_path)


# ============================================================
# Verify nbins
# ============================================================

successful_rows = (
    reclassification_report.loc[
        successful_mask
    ].copy()
)

if not successful_rows.empty:
    nbins_match = (
        successful_rows["new_nbins"]
        == successful_rows["expected_nbins"]
    )

    print(
        "\nCorrect Seitz nbins:",
        int(nbins_match.sum()),
        "/",
        len(successful_rows),
    )


# ============================================================
# Show changed classifications
# ============================================================

changed_states = (
    reclassification_report.loc[
        changed_mask,
        [
            "simulation_kind",
            "n_fcc_cells",
            "target_rho",
            "actual_rho",
            "kT",
            "old_nbins",
            "new_nbins",
            "old_phase_separated",
            "new_phase_separated",
            "old_low_density_fraction",
            "new_low_density_fraction",
            "state_path",
            "log_path",
        ],
    ]
    .sort_values(
        [
            "simulation_kind",
            "n_fcc_cells",
            "kT",
            "target_rho",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\nChanged classifications:",
    len(changed_states),
)

display(changed_states)


# ============================================================
# Show remaining failures
# ============================================================

failed_states = (
    reclassification_report.loc[
        failed_mask,
        [
            "simulation_kind",
            "n_fcc_cells",
            "state_path",
            "log_path",
            "error",
        ],
    ]
    .reset_index(drop=True)
)

if not failed_states.empty:
    print("\nMost common remaining errors:")

    display(
        failed_states["error"]
        .value_counts()
        .rename("count")
        .to_frame()
        .head(20)
    )

    print("\nFirst failed logs:")

    with pd.option_context(
        "display.max_rows",
        50,
        "display.max_columns",
        None,
        "display.max_colwidth",
        None,
    ):
        display(failed_states.head(50))

Classification module:
/home/pnichols/MDsims/md_Helpers/classification.py
thermalized: 1154 logs
cavitation_evolved: 674 logs
excitation_evolved: 37 logs

Total logs: 1865
Processed 25/1865 (failures: 0)
Processed 50/1865 (failures: 0)
Processed 75/1865 (failures: 1)
Processed 100/1865 (failures: 1)
Processed 125/1865 (failures: 1)
Processed 150/1865 (failures: 1)
Processed 175/1865 (failures: 1)
Processed 200/1865 (failures: 1)
Processed 225/1865 (failures: 1)
Processed 250/1865 (failures: 1)
Processed 275/1865 (failures: 1)
Processed 300/1865 (failures: 1)
Processed 325/1865 (failures: 1)
Processed 350/1865 (failures: 1)
Processed 375/1865 (failures: 1)
Processed 400/1865 (failures: 1)
Processed 425/1865 (failures: 1)
Processed 450/1865 (failures: 1)
Processed 475/1865 (failures: 1)
Processed 500/1865 (failures: 1)
Processed 525/1865 (failures: 2)
Processed 550/1865 (failures: 2)
Processed 575/1865 (failures: 2)
Processed 600/1865 (failures: 2)
Processed 625/1865 (failures: 2)
Proces

,simulation_kind,n_fcc_cells,target_rho,actual_rho,kT,old_nbins,new_nbins,old_phase_separated,new_phase_separated,old_low_density_fraction,new_low_density_fraction,state_path,log_path
0,excitation_evolved,60.0,0.71,0.71,0.8,10.0,21.0,False,True,0.01,0.018465,/exp/e961/data/MDsims-data/pnichols/Excitation...,/exp/e961/data/MDsims-data/pnichols/Excitation...
1,thermalized,10.0,0.68,0.68,0.8,NaN,6.0,NaN,True,NaN,0.027778,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...



Most common remaining errors:


,count
error,
KeyError('metadata/state does not contain n_fcc_cells'),3
OSError('Unable to synchronously open file (bad object header version number)'),1



First failed logs:


,simulation_kind,n_fcc_cells,state_path,log_path,error
0,thermalized,NaN,,/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.710/kT_0.700/nsteps_1000000/seed_1/randomization_log.hdf5,KeyError('metadata/state does not contain n_fcc_cells')
1,thermalized,NaN,,/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_20/rho_0.850/kT_0.900/nsteps_1000000/seed_1/randomization_log.hdf5,KeyError('metadata/state does not contain n_fcc_cells')
2,thermalized,NaN,,/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_28/rho_0.720/kT_0.720/nsteps_1000000/seed_1/randomization_log.hdf5,OSError('Unable to synchronously open file (bad object header version number)')
3,thermalized,NaN,,/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.705/kT_0.700/nsteps_1000000/seed_1/randomization_log.hdf5,KeyError('metadata/state does not contain n_fcc_cells')


In [2]:
missing_n_fcc_cells = (
    reclassification_report.loc[
        reclassification_report["error"].str.contains(
            "does not contain n_fcc_cells",
            na=False,
        ),
        [
            "simulation_kind",
            "state_path",
            "log_path",
            "error",
        ],
    ]
    .reset_index(drop=True)
)

display(missing_n_fcc_cells)

,simulation_kind,state_path,log_path,error
0,thermalized,,/exp/e961/data/MDsims-data/pnichols/Thermalize...,KeyError('metadata/state does not contain n_fc...
1,thermalized,,/exp/e961/data/MDsims-data/pnichols/Thermalize...,KeyError('metadata/state does not contain n_fc...
2,thermalized,,/exp/e961/data/MDsims-data/pnichols/Thermalize...,KeyError('metadata/state does not contain n_fc...


In [3]:
# ============================================================
# Recover n_fcc_cells for legacy v3 logs and recalculate their
# voxel phase-separation metadata using the Seitz nbins rule.
# ============================================================

from pathlib import Path
import re

import gsd.hoomd
import numpy as np
import pandas as pd

from md_Helpers import classification
from md_Helpers import metadata
from md_Helpers.paths import MASTER_CSVS_V3_ROOT


# ============================================================
# Settings
# ============================================================

DRY_RUN = False
BACKFILL_STATE_METADATA = True

density_threshold = (
    classification.DEFAULT_PHASE_SEP_DENSITY_THRESHOLD
)

voxel_fraction_threshold = (
    classification.DEFAULT_PHASE_SEP_VOXEL_FRACTION_THRESHOLD
)

legacy_report_path = (
    Path(MASTER_CSVS_V3_ROOT)
    / "legacy_missing_ncells_reclassification.csv"
)


# ============================================================
# Helpers
# ============================================================

def seitz_nbins(n_fcc_cells):
    return round(
        0.3 * int(n_fcc_cells) + 3
    )


def n_cells_from_text(value):
    """Extract n_cells_XX from a path or other text."""

    if value is None:
        return None

    match = re.search(
        r"(?:^|[/_])n_cells_(\d+)(?:[/_]|$)",
        str(value),
    )

    if match:
        return int(match.group(1))

    return None


def n_cells_from_fcc_particle_count(value):
    """
    Infer n from N = 4*n^3, accepting only an exact FCC count.
    """

    if value is None:
        return None

    try:
        N = int(value)
    except Exception:
        return None

    estimate = int(
        round((N / 4.0) ** (1.0 / 3.0))
    )

    if estimate > 0 and 4 * estimate**3 == N:
        return estimate

    return None


def n_cells_from_geometry(attrs):
    """
    Infer n_fcc_cells from BoxLength / fcc_cell_size.
    """

    box_length = attrs.get("BoxLength")
    cell_size = attrs.get("fcc_cell_size")

    if box_length is None or cell_size is None:
        return None

    try:
        ratio = float(box_length) / float(cell_size)
        estimate = int(round(ratio))
    except Exception:
        return None

    if estimate > 0 and np.isclose(
        ratio,
        estimate,
        rtol=0,
        atol=1e-6,
    ):
        return estimate

    return None


def infer_n_fcc_cells(log_path):
    """
    Recover n_fcc_cells using increasingly indirect evidence.
    """

    state_attrs = metadata.read_attrs(
        log_path,
        "metadata/state",
    )

    source_attrs = metadata.read_attrs(
        log_path,
        "metadata/source",
    )

    paths_attrs = metadata.read_attrs(
        log_path,
        "metadata/paths",
    )

    # Some older records stored attributes directly on metadata.
    legacy_attrs = metadata.read_attrs(
        log_path,
        "metadata",
    )

    # 1. Direct metadata values.
    direct_candidates = [
        (
            state_attrs.get("n_fcc_cells"),
            "metadata/state:n_fcc_cells",
        ),
        (
            source_attrs.get("source_n_fcc_cells"),
            "metadata/source:source_n_fcc_cells",
        ),
        (
            source_attrs.get("n_fcc_cells"),
            "metadata/source:n_fcc_cells",
        ),
        (
            legacy_attrs.get("n_fcc_cells"),
            "metadata:n_fcc_cells",
        ),
    ]

    for value, source in direct_candidates:
        if value is not None:
            return int(value), source

    # 2. Parse n_cells_XX from recorded paths.
    text_candidates = [
        ("log_path", log_path),
    ]

    text_candidates.extend(
        (
            f"metadata/paths:{key}",
            value,
        )
        for key, value in paths_attrs.items()
    )

    text_candidates.extend(
        (
            f"metadata/source:{key}",
            value,
        )
        for key, value in source_attrs.items()
    )

    for source, value in text_candidates:
        inferred = n_cells_from_text(value)

        if inferred is not None:
            return inferred, source

    # 3. Read the source thermalization log when available.
    source_log_value = source_attrs.get(
        "source_log_path"
    )

    if source_log_value:
        source_log_path = Path(source_log_value)

        if source_log_path.exists():
            source_state_attrs = metadata.read_attrs(
                source_log_path,
                "metadata/state",
            )

            source_n = source_state_attrs.get(
                "n_fcc_cells"
            )

            if source_n is not None:
                return (
                    int(source_n),
                    "source_log:metadata/state:n_fcc_cells",
                )

            inferred = n_cells_from_geometry(
                source_state_attrs
            )

            if inferred is not None:
                return (
                    inferred,
                    "source_log:BoxLength/fcc_cell_size",
                )

            inferred = n_cells_from_fcc_particle_count(
                source_state_attrs.get("N")
            )

            if inferred is not None:
                return (
                    inferred,
                    "source_log:FCC_particle_count",
                )

    # 4. Infer from this run's geometry.
    inferred = n_cells_from_geometry(
        state_attrs
    )

    if inferred is not None:
        return (
            inferred,
            "metadata/state:BoxLength/fcc_cell_size",
        )

    # 5. Infer from an exact FCC particle count.
    #
    # This will work for thermalized and unchanged-particle-count
    # excitation runs. It intentionally rejects cavitation states
    # whose particle counts no longer equal 4*n^3.
    inferred = n_cells_from_fcc_particle_count(
        state_attrs.get("N")
    )

    if inferred is not None:
        return (
            inferred,
            "metadata/state:FCC_particle_count",
        )

    return None, "not_inferred"


def find_saved_state(log_path):
    paths_attrs = metadata.read_attrs(
        log_path,
        "metadata/paths",
    )

    candidates = []

    for key in [
        "final_state_path",
        "state_path",
        "trajectory_path",
    ]:
        value = paths_attrs.get(key)

        if value:
            candidates.append(Path(value))

    candidates.extend(
        [
            log_path.with_name(
                log_path.name.replace(
                    "_log.hdf5",
                    ".gsd",
                )
            ),
            log_path.with_name(
                log_path.name.replace(
                    "_log.hdf5",
                    "_final.gsd",
                )
            ),
            log_path.with_name(
                log_path.name.replace(
                    "_log.hdf5",
                    "_trajectory.gsd",
                )
            ),
        ]
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


# ============================================================
# Select only the 10 missing-n_fcc_cells failures
# ============================================================

missing_mask = (
    reclassification_report["error"]
    .str.contains(
        "does not contain n_fcc_cells",
        na=False,
    )
)

legacy_logs = (
    reclassification_report.loc[
        missing_mask,
        [
            "simulation_kind",
            "log_path",
        ],
    ]
    .drop_duplicates("log_path")
    .reset_index(drop=True)
)

print(
    "Legacy logs to recover:",
    len(legacy_logs),
)


# ============================================================
# Recover and reclassify
# ============================================================

legacy_rows = []

for _, record in legacy_logs.iterrows():
    simulation_kind = record["simulation_kind"]
    log_path = Path(record["log_path"])

    result_row = {
        "status": "failed",
        "simulation_kind": simulation_kind,
        "n_fcc_cells": np.nan,
        "n_fcc_cells_source": "",
        "nbins": np.nan,
        "old_phase_separated": np.nan,
        "new_phase_separated": np.nan,
        "classification_changed": False,
        "state_path": "",
        "log_path": str(log_path),
        "error": "",
    }

    try:
        n_fcc_cells, inference_source = (
            infer_n_fcc_cells(log_path)
        )

        if n_fcc_cells is None:
            raise RuntimeError(
                "Could not safely infer n_fcc_cells"
            )

        nbins = seitz_nbins(
            n_fcc_cells
        )

        state_path = find_saved_state(
            log_path
        )

        if state_path is None:
            raise FileNotFoundError(
                "Could not find a saved state or trajectory"
            )

        old_voxel, _ = (
            classification.read_phase_method_attrs(
                log_path,
                "voxel",
            )
        )

        old_phase = old_voxel.get(
            "phase_separated",
            None,
        )

        with gsd.hoomd.open(
            name=str(state_path),
            mode="r",
        ) as trajectory:
            if len(trajectory) == 0:
                raise ValueError(
                    f"GSD contains no frames: {state_path}"
                )

            final_frame = trajectory[-1]

            new_voxel = (
                classification
                .compute_voxel_fraction_phase_separation(
                    final_frame,
                    nbins=nbins,
                    density_threshold=density_threshold,
                    voxel_fraction_threshold=(
                        voxel_fraction_threshold
                    ),
                )
            )

        new_voxel = dict(new_voxel)

        new_voxel.update(
            {
                "nbins": int(nbins),
                "nbins_source": "n_fcc_cells_rule",
                "n_fcc_cells": int(n_fcc_cells),
                "n_fcc_cells_source": inference_source,
                "updated_from_saved_gsd": True,
            }
        )

        new_phase = bool(
            new_voxel["phase_separated"]
        )

        if not DRY_RUN:
            # Backfill the missing state metadata so future tools can
            # use the Seitz rule without repeating this inference.
            if BACKFILL_STATE_METADATA:
                metadata.write_metadata_groups(
                    hdf5_path=log_path,
                    groups={
                        "metadata/state": {
                            "n_fcc_cells": int(
                                n_fcc_cells
                            ),
                        },
                    },
                    mode="a",
                    overwrite=True,
                )

            classification.write_phase_method_metadata(
                log_path=log_path,
                method_name="voxel",
                attrs=new_voxel,
                set_main_phase_separated=new_phase,
            )

        result_row.update(
            {
                "status": (
                    "dry_run"
                    if DRY_RUN
                    else "updated"
                ),
                "n_fcc_cells": n_fcc_cells,
                "n_fcc_cells_source": (
                    inference_source
                ),
                "nbins": nbins,
                "old_phase_separated": (
                    old_phase
                    if old_phase is not None
                    else np.nan
                ),
                "new_phase_separated": new_phase,
                "classification_changed": (
                    old_phase != new_phase
                ),
                "state_path": str(state_path),
            }
        )

    except Exception as error:
        result_row["error"] = repr(error)

    legacy_rows.append(result_row)


# ============================================================
# Results
# ============================================================

legacy_reclassification_report = pd.DataFrame(
    legacy_rows
)

legacy_reclassification_report.to_csv(
    legacy_report_path,
    index=False,
)

print("\nStatus counts:")
print(
    legacy_reclassification_report["status"]
    .value_counts(dropna=False)
)

print("\nRecovered box sizes:")
display(
    legacy_reclassification_report[
        "n_fcc_cells"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename("count")
    .to_frame()
)

print("\nDetailed results:")

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    120,
):
    display(
        legacy_reclassification_report
    )

print("\nSaved legacy audit report:")
print(legacy_report_path)

Legacy logs to recover: 3

Status counts:
status
failed    3
Name: count, dtype: int64

Recovered box sizes:


,count
n_fcc_cells,
NaN,3



Detailed results:


,status,simulation_kind,n_fcc_cells,n_fcc_cells_source,nbins,old_phase_separated,new_phase_separated,classification_changed,state_path,log_path,error
0,failed,thermalized,NaN,,NaN,NaN,NaN,False,,/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.710/kT_0.700/nsteps_1000000/seed_1/ra...,FileNotFoundError('Could not find a saved state or trajectory')
1,failed,thermalized,NaN,,NaN,NaN,NaN,False,,/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_20/rho_0.850/kT_0.900/nsteps_1000000/seed_1/ra...,FileNotFoundError('Could not find a saved state or trajectory')
2,failed,thermalized,NaN,,NaN,NaN,NaN,False,,/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.705/kT_0.700/nsteps_1000000/seed_1/ra...,FileNotFoundError('Could not find a saved state or trajectory')



Saved legacy audit report:
/exp/e961/data/MDsims-data/pnichols/Master_CSVs_v3/legacy_missing_ncells_reclassification.csv


In [4]:
import h5py
import pandas as pd

layout_rows = []

for _, record in legacy_logs.iterrows():
    log_path = Path(record["log_path"])

    row = {
        "simulation_kind": record["simulation_kind"],
        "log_path": str(log_path),
        "metadata_layout": "unknown",
        "data_version": None,
        "state_kind": None,
        "n_fcc_cells_v3": None,
        "n_fcc_cells_legacy": None,
        "has_v3_state_group": False,
        "has_v3_classification_group": False,
        "has_legacy_phase_group": False,
        "error": "",
    }

    try:
        with h5py.File(
            log_path,
            mode="r",
        ) as hdf:
            metadata_group = hdf.get("metadata")

            if metadata_group is None:
                row["metadata_layout"] = "no_metadata"

            else:
                # Legacy/v2-style metadata attributes.
                row["n_fcc_cells_legacy"] = (
                    metadata_group.attrs.get(
                        "n_fcc_cells",
                        None,
                    )
                )

                legacy_data_version = (
                    metadata_group.attrs.get(
                        "data_version",
                        None,
                    )
                )

                # Canonical v3 metadata/state group.
                if "metadata/state" in hdf:
                    row["has_v3_state_group"] = True

                    state_group = hdf[
                        "metadata/state"
                    ]

                    row["n_fcc_cells_v3"] = (
                        state_group.attrs.get(
                            "n_fcc_cells",
                            None,
                        )
                    )

                    row["data_version"] = (
                        state_group.attrs.get(
                            "data_version",
                            legacy_data_version,
                        )
                    )

                    row["state_kind"] = (
                        state_group.attrs.get(
                            "state_kind",
                            None,
                        )
                    )

                else:
                    row["data_version"] = (
                        legacy_data_version
                    )

                row[
                    "has_v3_classification_group"
                ] = (
                    "metadata/classification/"
                    "phase_separation"
                    in hdf
                )

                row[
                    "has_legacy_phase_group"
                ] = (
                    "metadata/phase_separation"
                    in hdf
                )

                # Determine the likely layout.
                if (
                    row["has_v3_state_group"]
                    and row["n_fcc_cells_v3"]
                    is not None
                ):
                    row["metadata_layout"] = (
                        "canonical_v3"
                    )

                elif (
                    row["n_fcc_cells_legacy"]
                    is not None
                ):
                    row["metadata_layout"] = (
                        "legacy_or_v2_layout"
                    )

                elif row["has_v3_state_group"]:
                    row["metadata_layout"] = (
                        "incomplete_v3_or_migrated"
                    )

                else:
                    row["metadata_layout"] = (
                        "unrecognized_legacy_layout"
                    )

    except Exception as error:
        row["metadata_layout"] = "unreadable"
        row["error"] = repr(error)

    layout_rows.append(row)

layout_report = pd.DataFrame(layout_rows)

display(
    layout_report[
        [
            "simulation_kind",
            "metadata_layout",
            "data_version",
            "state_kind",
            "n_fcc_cells_v3",
            "n_fcc_cells_legacy",
            "has_v3_state_group",
            "has_v3_classification_group",
            "has_legacy_phase_group",
            "log_path",
            "error",
        ]
    ]
)

,simulation_kind,metadata_layout,data_version,state_kind,n_fcc_cells_v3,n_fcc_cells_legacy,has_v3_state_group,has_v3_classification_group,has_legacy_phase_group,log_path,error
0,thermalized,no_metadata,None,None,None,None,False,False,False,/exp/e961/data/MDsims-data/pnichols/Thermalize...,
1,thermalized,no_metadata,None,None,None,None,False,False,False,/exp/e961/data/MDsims-data/pnichols/Thermalize...,
2,thermalized,no_metadata,None,None,None,None,False,False,False,/exp/e961/data/MDsims-data/pnichols/Thermalize...,


In [5]:
verification_rows = []

for _, record in layout_report.iterrows():
    if record["metadata_layout"] != "canonical_v3":
        continue

    log_path = Path(record["log_path"])

    state_attrs = metadata.read_attrs(
        log_path,
        "metadata/state",
    )

    voxel_attrs, voxel_path = (
        classification.read_phase_method_attrs(
            log_path,
            "voxel",
        )
    )

    verification_rows.append(
        {
            "simulation_kind": record["simulation_kind"],
            "n_fcc_cells": state_attrs.get(
                "n_fcc_cells"
            ),
            "nbins": voxel_attrs.get("nbins"),
            "nbins_source": voxel_attrs.get(
                "nbins_source"
            ),
            "phase_separated": voxel_attrs.get(
                "phase_separated"
            ),
            "low_density_fraction": voxel_attrs.get(
                "low_density_fraction"
            ),
            "voxel_metadata_path": voxel_path,
            "log_path": str(log_path),
        }
    )

verification_report = pd.DataFrame(
    verification_rows
)

display(verification_report)

""


In [6]:
no_metadata_logs = layout_report.loc[
    layout_report["metadata_layout"] == "no_metadata",
    ["simulation_kind", "log_path"],
].copy()

inspection_rows = []

for _, record in no_metadata_logs.iterrows():
    log_path = Path(record["log_path"])

    state_candidates = [
        log_path.with_name(
            log_path.name.replace(
                "_log.hdf5",
                ".gsd",
            )
        ),
        log_path.with_name(
            log_path.name.replace(
                "_log.hdf5",
                "_final.gsd",
            )
        ),
    ]

    existing_states = [
        str(path)
        for path in state_candidates
        if path.exists()
    ]

    with h5py.File(log_path, mode="r") as hdf:
        inspection_rows.append(
            {
                "file_size_bytes": log_path.stat().st_size,
                "top_level_groups": list(hdf.keys()),
                "has_thermodynamic_data": (
                    "hoomd-data/md/compute/"
                    "ThermodynamicQuantities"
                    in hdf
                ),
                "existing_state_files": existing_states,
                "log_path": str(log_path),
            }
        )

no_metadata_inspection = pd.DataFrame(
    inspection_rows
)

with pd.option_context(
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
):
    display(no_metadata_inspection)

,file_size_bytes,top_level_groups,has_thermodynamic_data,existing_state_files,log_path
0,56464,[hoomd-data],True,[],/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.710/kT_0.700/nsteps_1000000/seed_1/randomization_log.hdf5
1,52384,[hoomd-data],True,[],/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_20/rho_0.850/kT_0.900/nsteps_1000000/seed_1/randomization_log.hdf5
2,60544,[hoomd-data],True,[],/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.705/kT_0.700/nsteps_1000000/seed_1/randomization_log.hdf5


In [7]:
from md_Helpers import simulation as sim

incomplete_runs = [
    {
        "n_fcc_cells": 10,
        "target_rho": 0.710,
        "kT": 0.700,
    },
    {
        "n_fcc_cells": 20,
        "target_rho": 0.850,
        "kT": 0.900,
    },
    {
        "n_fcc_cells": 30,
        "target_rho": 0.705,
        "kT": 0.700,
    },
]

for parameters in incomplete_runs:
    print("=" * 80)
    print("Rerunning:", parameters)

    result = sim.get_or_make_thermalized_state(
        **parameters,
        nsteps=1_000_000,
        seed=1,
        phase_name="randomization",
        overwrite=True,
        overwrite_lattice=False,
    )

    print("created_new:", result["created_new"])
    print("state:", result["paths"]["state_path"])
    print("log:", result["paths"]["log_path"])

Rerunning: {'n_fcc_cells': 10, 'target_rho': 0.71, 'kT': 0.7}
Loaded existing FCC lattice:
/exp/e961/data/MDsims-data/pnichols/Simple_Lattices_v3/FCC/n_cells_10/rho_0.710/lattice.gsd
Using GPU device
Final device: <hoomd.device.GPU object at 0x7f6480334590>
Started HDF5 logger
Log file: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.710/kT_0.700/nsteps_1000000/seed_1/randomization_log.hdf5
Log period: 1000
Stopped HDF5 logger
Wrote HDF5 metadata
Log file: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.710/kT_0.700/nsteps_1000000/seed_1/randomization_log.hdf5
Metadata group: metadata
Saved final state
GSD file: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.710/kT_0.700/nsteps_1000000/seed_1/randomization.gsd


TypeError: argument of type 'PosixPath' is not iterable